# Agentic Workflow Architecture: Async Research & Reporting Pipeline
**Framework:** OpenAI Agents SDK (`openai-agents`)
**Language:** Python
**Execution:** Asynchronous Parallel Processing

## 1. System Overview
This pipeline orchestrates a multi-agent system to automate a comprehensive research task. It leverages the OpenAI Agents SDK to manage states, enforce structured data schemas (via Pydantic), and execute tool calls.

The system uses a **Sequential-Parallel-Sequential** flow:
1. **Planning (Sequential):** A Planner Agent breaks a high-level topic into specific sub-queries.
2. **Execution (Parallel):** Multiple Search Agents independently query the web and summarize findings concurrently.
3. **Synthesis (Sequential):** A Writer Agent compiles the summarized data into a lengthy Markdown report.
4. **Dispatch (Sequential):** An Email Agent formats the report into HTML and dispatches it via a custom tool.

---

## 2. Agent Roles & Schemas

### A. Planner Agent
*   **Role:** Decomposes a user query into a targeted research plan.
*   **Output Constraint:** Forced to return a `WebSearchPlan` (Pydantic model) containing a list of `WebSearchItem` objects (reasoning + query).

### B. Search Agent
*   **Role:** Executes web searches and distills the raw results into concise (under 300 words) summaries.
*   **Tool:** Native `WebSearchTool()`.
*   **Execution Rule:** `tool_choice="required"` ensures the agent must search before answering.

### C. Writer Agent
*   **Role:** Ingests all search summaries and synthesizes a comprehensive 5-10 page report.
*   **Output Constraint:** Forced to return a `ReportData` object (summary + markdown report + follow-up questions).

### D. Email Agent
*   **Role:** Prepares the final report and routes it to users.
*   **Tool:** Custom `@function_tool` named `send_email_tool`.

---

## 3. Reconstructed Code Blueprint

Below is the clean, reconstructed codebase modularized for readability and deployment.

In [ ]:

import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# OpenAI Agents SDK
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings

# Assuming 'messenger' is a local/custom module for dispatch
from messenger import send_email, push 

load_dotenv(override=True)

# --- Configuration ---
MODEL_NAME = "gpt-5.4-mini" # Example model assignment
USE_EMAIL = True
HOW_MANY_SEARCHES = 5
TASK_QUERY = "Most popular AI Agent frameworks in 2026"

# ==========================================
# 1. DATA SCHEMAS (Pydantic)
# ==========================================
class WebSearchItem(BaseModel):
    reason: str = Field(description="Reasoning for why this search is important.")
    query: str = Field(description="The exact search term to query.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="List of web searches to perform.")

class ReportData(BaseModel):
    short_summary: str = Field(description="A 2-3 sentence summary of findings.")
    markdown_report: str = Field(description="The final comprehensive report.")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further.")


# ==========================================
# 2. CUSTOM TOOLS
# ==========================================
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """Send out an email with the given subject and body."""
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email dispatched successfully."


# ==========================================
# 3. AGENT DEFINITIONS
# ==========================================

# Agent 1: Planner
planner_agent = Agent(
    name="Planner Agent", 
    instructions=f"Given a query, formulate {HOW_MANY_SEARCHES} web searches to best answer it.", 
    model=MODEL_NAME, 
    output_type=WebSearchPlan
)

# Agent 2: Searcher
search_agent = Agent(
    name="Search Agent", 
    instructions=(
        "Search the web for the provided term and produce a concise summary. "
        "The summary must be 2-3 paragraphs, under 300 words. Reply only with the summary."
    ), 
    tools=[WebSearchTool()], 
    model=MODEL_NAME, 
    model_settings=ModelSettings(tool_choice="required")
)

# Agent 3: Writer
writer_agent = Agent(
    name="Writer Agent", 
    instructions=(
        "Write a cohesive, detailed report based on the provided research. "
        "The final output must be in markdown, highly detailed, aiming for 1000+ words."
    ), 
    model=MODEL_NAME, 
    output_type=ReportData
)

# Agent 4: Emailer
email_agent = Agent(
    name="Email Agent", 
    instructions="Convert the provided report into clean HTML and send an email with an appropriate subject.", 
    tools=[send_email_tool], 
    model=MODEL_NAME
)


# ==========================================
# 4. ASYNC ORCHESTRATION FUNCTIONS
# ==========================================
async def plan_and_search(query: str) -> list[str]:
    """Generates a search plan and executes searches concurrently."""
    print("[1] Planning searches...")
    plan_result = await Runner.run(planner_agent, f"Query: {query}")
    searches = plan_result.final_output.searches
    
    print(f"[2] Executing {len(searches)} concurrent searches...")
    tasks = [execute_single_search(item) for item in searches]
    return await asyncio.gather(*tasks)

async def execute_single_search(item: WebSearchItem) -> str:
    """Runs a single search task using the Search Agent."""
    input_message = f"Search term: {item.query}\nReason: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

async def synthesize_report(query: str, search_results: list[str]) -> ReportData:
    """Compiles raw results into a structured report."""
    print("[3] Synthesizing final report...")
    input_message = f"Original query: {query}\nSummarized results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    return result.final_output

async def dispatch_report(report: ReportData):
    """Triggers the Email Agent to dispatch the final document."""
    print("[4] Dispatching via email tool...")
    await Runner.run(email_agent, report.markdown_report)
    print("Email sent successfully!")


# ==========================================
# 5. MAIN EXECUTION LOOP
# ==========================================
async def main():
    with trace("Automated Research Pipeline"):
        print(f"Starting pipeline for task: '{TASK_QUERY}'\n" + "-"*40)
        
        # Step 1 & 2: Plan and execute parallel searches
        search_results = await plan_and_search(TASK_QUERY)
        
        # Step 3: Write report
        final_report = await synthesize_report(TASK_QUERY, search_results)
        
        # Step 4: Dispatch
        await dispatch_report(final_report)
        
        print("-" * 40 + "\nPipeline Complete. Hooray!")

if __name__ == "__main__":
    asyncio.run(main())